In [ ]:
import cv2
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import ParameterGrid
import warnings
warnings.filterwarnings('ignore')

DATASET_PATH = "/kaggle/input/datasets/angelinacandaya/dataset-compvis/dataset1"
OUTPUT_PATH = "/kaggle/working/output_v6"
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, "matches"), exist_ok=True)  

NFEATURES = 4000
LOWE_RATIO = 0.75
VALIDATION_SPLIT = 0.2

orb = cv2.ORB_create(nfeatures=NFEATURES)
sift = cv2.SIFT_create(nfeatures=NFEATURES)
bf_orb = cv2.BFMatcher(cv2.NORM_HAMMING)
bf_sift = cv2.BFMatcher(cv2.NORM_L2)

def load_gray(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Gambar tidak ditemukan: {path}")
    return img

def get_features(img, method):
    if method == "ORB":
        return orb.detectAndCompute(img, None)
    return sift.detectAndCompute(img, None)

def match_features(des1, des2, method):
    bf = bf_orb if method == "ORB" else bf_sift
    if des1 is None or des2 is None or len(des1) < 2 or len(des2) < 2:
        return []
    matches = bf.knnMatch(des1, des2, k=2)
    good = []
    for pair in matches:
        if len(pair) == 2:
            m, n = pair
            if m.distance < LOWE_RATIO * n.distance:
                good.append(m)
    return good

def hitung_score(good_matches, kp1, kp2):
    base = max(len(kp1), len(kp2))
    if base == 0:
        return 0.0
    return min(len(good_matches) / base * 100, 100.0)

def tune_thresholds(book, bd, valid_images, valid_labels):
    best_params = {}
    for method in ["ORB", "SIFT"]:
        best_acc = 0
        best_t_auth = 0.30
        best_t_susp = 0.08
        for t_auth in np.arange(0.25, 0.55, 0.03):
            for t_susp in np.arange(0.05, 0.20, 0.02):
                if t_susp >= t_auth:
                    continue
                correct = 0
                for img_path, true_label in valid_images:
                    test_img = load_gray(img_path)
                    if method == "ORB":
                        kp_test, des_test = get_features(test_img, "ORB")
                        good = match_features(bd["des_orb"], des_test, "ORB")
                        score = hitung_score(good, bd["kp_orb"], kp_test)
                        ratio = score / bd["baseline_orb"]
                    else:
                        kp_test, des_test = get_features(test_img, "SIFT")
                        good = match_features(bd["des_sift"], des_test, "SIFT")
                        score = hitung_score(good, bd["kp_sift"], kp_test)
                        ratio = score / bd["baseline_sift"]
                    
                    if ratio >= t_auth:
                        pred = "AUTHENTIC"
                    elif ratio >= t_susp:
                        pred = "SUSPICIOUS"
                    else:
                        pred = "COUNTERFEIT"
                    
                    if pred == true_label:
                        correct += 1
                acc = correct / len(valid_images)
                if acc > best_acc:
                    best_acc = acc
                    best_t_auth = t_auth
                    best_t_susp = t_susp
        best_params[method] = (best_t_auth, best_t_susp)
        print(f"    {method}: t_auth={best_t_auth:.2f}, t_susp={best_t_susp:.2f} (acc val={best_acc:.2f})")
    return best_params

def ensemble_decision(label_orb, label_sift, ratio_orb, ratio_sift,
                      t_auth_orb, t_susp_orb, t_auth_sift, t_susp_sift):
    if label_orb == label_sift:
        return label_orb
    
    if (label_orb == "AUTHENTIC" and label_sift == "COUNTERFEIT") or \
       (label_orb == "COUNTERFEIT" and label_sift == "AUTHENTIC"):
        if label_orb == "AUTHENTIC":
            conf_orb = (ratio_orb - t_auth_orb) / (1 - t_auth_orb)
            conf_sift = (t_susp_sift - ratio_sift) / t_susp_sift if ratio_sift < t_susp_sift else 0
        else:
            conf_orb = (t_susp_orb - ratio_orb) / t_susp_orb if ratio_orb < t_susp_orb else 0
            conf_sift = (ratio_sift - t_auth_sift) / (1 - t_auth_sift)
        if (label_orb == "AUTHENTIC" and conf_orb > conf_sift) or \
           (label_sift == "AUTHENTIC" and conf_sift > conf_orb):
            return "SUSPICIOUS"
        else:
            return "COUNTERFEIT"
    
    if (label_orb == "AUTHENTIC" and label_sift == "SUSPICIOUS") or \
       (label_orb == "SUSPICIOUS" and label_sift == "AUTHENTIC"):
        return "SUSPICIOUS"
    
    if (label_orb == "SUSPICIOUS" and label_sift == "COUNTERFEIT") or \
       (label_orb == "COUNTERFEIT" and label_sift == "SUSPICIOUS"):
        if label_orb == "COUNTERFEIT" and ratio_orb < t_susp_orb * 0.8:
            return "COUNTERFEIT"
        if label_sift == "COUNTERFEIT" and ratio_sift < t_susp_sift * 0.8:
            return "COUNTERFEIT"
        return "SUSPICIOUS"
    
    return "SUSPICIOUS"

def save_matches_visualization(ref_img, kp_ref_orb, kp_ref_sift, test_img, kp_test_orb, kp_test_sift,
                               good_orb, good_sift, title, save_path):
    """
    Menyimpan gambar dengan dua subplot: ORB matches (kiri) dan SIFT matches (kanan).
    Menampilkan maksimal 50 matches terbaik.
    """
    vis_orb = cv2.drawMatches(ref_img, kp_ref_orb, test_img, kp_test_orb,
                              good_orb[:50], None,
                              flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    vis_sift = cv2.drawMatches(ref_img, kp_ref_sift, test_img, kp_test_sift,
                               good_sift[:50], None,
                               flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle(title, fontsize=10, fontweight='bold')
    axes[0].imshow(vis_orb, cmap='gray')
    axes[0].set_title(f"ORB -- {len(good_orb)} matches", fontsize=9)
    axes[0].axis('off')
    axes[1].imshow(vis_sift, cmap='gray')
    axes[1].set_title(f"SIFT -- {len(good_sift)} matches", fontsize=9)
    axes[1].axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.close()

print("=" * 70)
print("  BOOK COVER AUTHENTICITY - v6 (TUNED THRESHOLDS + VISUALIZATION)")
print("=" * 70)

books = {}
if not os.path.exists(DATASET_PATH):
    raise Exception(f"Dataset path tidak ditemukan: {DATASET_PATH}")

for book in sorted(os.listdir(DATASET_PATH)):
    bp = os.path.join(DATASET_PATH, book)
    if not os.path.isdir(bp):
        continue
    ref_path = os.path.join(bp, "reference", "asli.jpg")
    if not os.path.exists(ref_path):
        print(f"[SKIP] {book} -> reference/asli.jpg missing")
        continue
    
    ref_img = load_gray(ref_path)
    kp_o, des_o = get_features(ref_img, "ORB")
    kp_s, des_s = get_features(ref_img, "SIFT")
    
    baseline_path = os.path.join(bp, "test", "brightness_normal.jpg")
    baseline_orb, baseline_sift = 100.0, 100.0
    if os.path.exists(baseline_path):
        bl_img = load_gray(baseline_path)
        kp_bl_o, des_bl_o = get_features(bl_img, "ORB")
        kp_bl_s, des_bl_s = get_features(bl_img, "SIFT")
        g_o = match_features(des_o, des_bl_o, "ORB")
        g_s = match_features(des_s, des_bl_s, "SIFT")
        baseline_orb = max(hitung_score(g_o, kp_o, kp_bl_o), 1.0)
        baseline_sift = max(hitung_score(g_s, kp_s, kp_bl_s), 1.0)
    
    books[book] = {
        "ref_img": ref_img,
        "kp_orb": kp_o, "des_orb": des_o,
        "kp_sift": kp_s, "des_sift": des_s,
        "baseline_orb": baseline_orb,
        "baseline_sift": baseline_sift,
        "path": bp,
    }
    print(f"[LOADED] {book}: ORB kp={len(kp_o)}, SIFT kp={len(kp_s)} | baseline ORB={baseline_orb:.1f}% SIFT={baseline_sift:.1f}%")

print(f"\nTotal buku: {len(books)}\n")

all_results = []

for book, bd in books.items():
    print(f"\n--- Processing {book} ---")
    
    test_folder = os.path.join(bd["path"], "test")
    fake_folder = os.path.join(bd["path"], "fake")
    
    valid_images = []
    if os.path.isdir(test_folder):
        for f in os.listdir(test_folder):
            if f.lower().endswith(('.jpg','.jpeg','.png')):
                valid_images.append((os.path.join(test_folder, f), "AUTHENTIC"))
    if os.path.isdir(fake_folder):
        for f in os.listdir(fake_folder):
            if f.lower().endswith(('.jpg','.jpeg','.png')):
                valid_images.append((os.path.join(fake_folder, f), "SUSPICIOUS"))
    
    if len(valid_images) < 5:
        print(f"  Warning: hanya {len(valid_images)} sampel, gunakan threshold default.")
        t_auth_orb, t_susp_orb = 0.35, 0.08
        t_auth_sift, t_susp_sift = 0.35, 0.08
        train_images = []
        val_images = valid_images
    else:
        np.random.seed(42)
        indices = np.random.permutation(len(valid_images))
        split = int(VALIDATION_SPLIT * len(valid_images))
        val_indices = indices[:split]
        train_indices = indices[split:]
        
        train_images = [valid_images[i] for i in train_indices]
        val_images = [valid_images[i] for i in val_indices]
        
        print(f"  Tuning threshold menggunakan {len(train_images)} sampel (train), validasi {len(val_images)} sampel")
        best_params = tune_thresholds(book, bd, train_images, [label for _, label in train_images])
        t_auth_orb, t_susp_orb = best_params["ORB"]
        t_auth_sift, t_susp_sift = best_params["SIFT"]
    
    for img_path, true_label in val_images:
        test_img = load_gray(img_path)
        fname = os.path.basename(img_path)
        
        # ORB
        kp_to, des_to = get_features(test_img, "ORB")
        g_o = match_features(bd["des_orb"], des_to, "ORB")
        score_o = hitung_score(g_o, bd["kp_orb"], kp_to)
        ratio_o = score_o / bd["baseline_orb"]
        if ratio_o >= t_auth_orb:
            label_o = "AUTHENTIC"
        elif ratio_o >= t_susp_orb:
            label_o = "SUSPICIOUS"
        else:
            label_o = "COUNTERFEIT"
        
        # SIFT
        kp_ts, des_ts = get_features(test_img, "SIFT")
        g_s = match_features(bd["des_sift"], des_ts, "SIFT")
        score_s = hitung_score(g_s, bd["kp_sift"], kp_ts)
        ratio_s = score_s / bd["baseline_sift"]
        if ratio_s >= t_auth_sift:
            label_s = "AUTHENTIC"
        elif ratio_s >= t_susp_sift:
            label_s = "SUSPICIOUS"
        else:
            label_s = "COUNTERFEIT"
        
        label_ens = ensemble_decision(label_o, label_s, ratio_o, ratio_s,
                                      t_auth_orb, t_susp_orb, t_auth_sift, t_susp_sift)
        
        all_results.append({
            "Reference": book,
            "Test": fname,
            "True": true_label,
            "ORB_Label": label_o,
            "SIFT_Label": label_s,
            "ENS_Label": label_ens,
            "ORB_Ratio": ratio_o,
            "SIFT_Ratio": ratio_s,
        })
        
        vis_title = f"{book} vs {fname} | ORB:{label_o} ({score_o:.1f}%) SIFT:{label_s} ({score_s:.1f}%) ENS:{label_ens}"
        vis_filename = f"{book}_{fname.replace('.jpg','')}_match.jpg"
        vis_path = os.path.join(OUTPUT_PATH, "matches", vis_filename)
        save_matches_visualization(bd["ref_img"], bd["kp_orb"], bd["kp_sift"],
                                   test_img, kp_to, kp_ts,
                                   g_o, g_s, vis_title, vis_path)
    
        for other_book, other_bd in books.items():
        if other_book == book:
            continue
        test_img = other_bd["ref_img"]
        fname = f"cross_{other_book}"
        
        # ORB
        kp_to, des_to = get_features(test_img, "ORB")
        g_o = match_features(bd["des_orb"], des_to, "ORB")
        score_o = hitung_score(g_o, bd["kp_orb"], kp_to)
        ratio_o = score_o / bd["baseline_orb"]
        label_o = "COUNTERFEIT" if ratio_o < t_susp_orb else ("SUSPICIOUS" if ratio_o < t_auth_orb else "AUTHENTIC")
        
        # SIFT
        kp_ts, des_ts = get_features(test_img, "SIFT")
        g_s = match_features(bd["des_sift"], des_ts, "SIFT")
        score_s = hitung_score(g_s, bd["kp_sift"], kp_ts)
        ratio_s = score_s / bd["baseline_sift"]
        label_s = "COUNTERFEIT" if ratio_s < t_susp_sift else ("SUSPICIOUS" if ratio_s < t_auth_sift else "AUTHENTIC")
        
        label_ens = ensemble_decision(label_o, label_s, ratio_o, ratio_s,
                                      t_auth_orb, t_susp_orb, t_auth_sift, t_susp_sift)
        
        all_results.append({
            "Reference": book,
            "Test": fname,
            "True": "COUNTERFEIT",
            "ORB_Label": label_o,
            "SIFT_Label": label_s,
            "ENS_Label": label_ens,
            "ORB_Ratio": ratio_o,
            "SIFT_Ratio": ratio_s,
        })
        
        vis_title = f"{book} vs {other_book} (COUNTERFEIT) | ORB:{label_o} SIFT:{label_s} ENS:{label_ens}"
        vis_filename = f"{book}_vs_{other_book}_cross.jpg"
        vis_path = os.path.join(OUTPUT_PATH, "matches", vis_filename)
        save_matches_visualization(bd["ref_img"], bd["kp_orb"], bd["kp_sift"],
                                   test_img, kp_to, kp_ts,
                                   g_o, g_s, vis_title, vis_path)
    
    print(f"  Selesai {book}, threshold ORB=({t_auth_orb:.2f},{t_susp_orb:.2f}) SIFT=({t_auth_sift:.2f},{t_susp_sift:.2f})")

# Evaluasi
df = pd.DataFrame(all_results)
print("\n" + "=" * 70)
print("  HASIL EVALUASI v6 (DENGAN VISUALISASI MATCHING)")
print("=" * 70)

for label in ["AUTHENTIC", "SUSPICIOUS", "COUNTERFEIT"]:
    sub = df[df["True"] == label]
    if len(sub) > 0:
        acc_ens = (sub["ENS_Label"] == label).sum() / len(sub) * 100
        print(f"{label:12} | Ensemble acc: {acc_ens:.1f}% ({ (sub['ENS_Label']==label).sum()}/{len(sub)})")

total_acc = (df["ENS_Label"] == df["True"]).sum() / len(df) * 100
print(f"\nOverall Ensemble Accuracy: {total_acc:.2f}% ({ (df['ENS_Label']==df['True']).sum()}/{len(df)})")

print("\nClassification Report (Ensemble):")
print(classification_report(df["True"], df["ENS_Label"], labels=["AUTHENTIC","SUSPICIOUS","COUNTERFEIT"]))

df.to_csv(os.path.join(OUTPUT_PATH, "hasil_v6.csv"), index=False)

labels_order = ["AUTHENTIC", "SUSPICIOUS", "COUNTERFEIT"]
cm = confusion_matrix(df["True"], df["ENS_Label"], labels=labels_order)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_order)
disp.plot(cmap="Blues")
plt.title(f"Confusion Matrix - v6 (Acc {total_acc:.1f}%)")
plt.savefig(os.path.join(OUTPUT_PATH, "confusion_matrix_v6.png"), dpi=120, bbox_inches="tight")
plt.close()

print(f"\nOutput saved to {OUTPUT_PATH}/")
print(f"  - CSV: hasil_v6.csv")
print(f"  - Confusion matrix: confusion_matrix_v6.png")
print(f"  - Semua gambar matching disimpan di folder 'matches/'")
print("=" * 70)

  BOOK COVER AUTHENTICITY - v6 (TUNED THRESHOLDS + VISUALIZATION)
[LOADED] Atomic Habits: ORB kp=4000, SIFT kp=2258 | baseline ORB=97.5% SIFT=89.6%
[LOADED] Berani Tidak Disukai: ORB kp=4000, SIFT kp=1558 | baseline ORB=97.4% SIFT=93.1%
[LOADED] Filosofi Teras: ORB kp=4000, SIFT kp=4000 | baseline ORB=97.0% SIFT=94.8%
[LOADED] Pasien: ORB kp=3435, SIFT kp=4000 | baseline ORB=95.5% SIFT=92.7%
[LOADED] Pride and Prejudice: ORB kp=3410, SIFT kp=1102 | baseline ORB=96.0% SIFT=93.5%
[LOADED] Sang Alkemis: ORB kp=3930, SIFT kp=993 | baseline ORB=94.3% SIFT=88.6%
[LOADED] Seni Bersikap Bodoamat: ORB kp=4000, SIFT kp=1441 | baseline ORB=96.7% SIFT=89.2%
[LOADED] Teka Teki Gambar Aneh: ORB kp=3837, SIFT kp=988 | baseline ORB=94.0% SIFT=90.9%
[LOADED] Teka Teki Rumah Aneh: ORB kp=2280, SIFT kp=510 | baseline ORB=97.3% SIFT=93.6%
[LOADED] Toko Kelontong Namiya: ORB kp=3821, SIFT kp=4000 | baseline ORB=97.1% SIFT=94.2%

Total buku: 10


--- Processing Atomic Habits ---
  Tuning threshold menggunak